# 01 — First Backtest: LightGBM + Alpha158 on S&P 500

This notebook walks through a complete quant research cycle:

1. **Setup** — install deps, download data, initialize QLib
2. **Train** — fit a LightGBM model on 158 technical alpha factors
3. **Backtest** — simulate a portfolio strategy on out-of-sample data
4. **Analyze** — visualize returns, drawdowns, and signal quality

### What is Alpha158?
Alpha158 is QLib's built-in feature set of 158 hand-crafted technical indicators
(moving averages, RSI, MACD, volume ratios, etc.). It's a strong baseline for
stock return prediction.

### What is LightGBM?
A gradient-boosted decision tree model. Fast to train, handles tabular data well,
and is a common first choice in quant ML pipelines.

## Step 1 — Environment Setup

Run the shared setup notebook to install dependencies, download data, and
initialize QLib. This is idempotent — safe to re-run.

In [ ]:
%run ../setup.ipynb

## Step 2 — Load the Strategy Config

Our strategy is defined in two places:
- `configs/lgbm_alpha158.yaml` — the QLib workflow config (model hyperparams,
  data splits, portfolio rules)
- `strategies/lgbm_alpha158.py` — Python code that orchestrates training,
  prediction, and backtesting

Let's peek at the config to understand what we're running.

In [ ]:
import yaml

with open("configs/lgbm_alpha158.yaml") as f:
    config = yaml.safe_load(f)

print("Model:", config["task"]["model"]["class"])
print("Features:", config["task"]["dataset"]["kwargs"]["handler"]["class"])
print("Universe:", config["task"]["dataset"]["kwargs"]["handler"]["kwargs"]["instruments"])
print("\nData Splits:")
for split, dates in config["task"]["dataset"]["kwargs"]["segments"].items():
    print(f"  {split}: {dates[0]} to {dates[1]}")
print(f"\nBacktest: {config['port_analysis_config']['backtest']['start_time']} to {config['port_analysis_config']['backtest']['end_time']}")
print(f"Benchmark: {config['port_analysis_config']['backtest']['benchmark']}")
print(f"Starting capital: ${config['port_analysis_config']['backtest']['account']:,.0f}")

## Step 3 — Run the Backtest

This single function call:
1. Builds the Alpha158 feature matrix for all S&P 500 stocks
2. Trains a LightGBM model on the training split (2008–2020)
3. Generates predictions on the test split (2023–2025)
4. Simulates a TopkDropout portfolio (hold top 50 stocks, rotate 5/day)
5. Computes IC, returns, Sharpe, max drawdown, and more

**This will take a few minutes** depending on your Colab runtime.

In [ ]:
from strategies.lgbm_alpha158 import run_backtest

# init=False because setup.ipynb already initialized QLib
results = run_backtest(init=False)

## Step 4 — Visualize Results

### 4a. Cumulative Returns

This plot shows the strategy's cumulative return vs. the benchmark (SPY).
- **Blue line** = strategy (excess return over benchmark)
- A rising line means the strategy is outperforming the market.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

report = results["report_normal"]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Cumulative return
axes[0].plot(report.index, report["return"].cumsum(), label="Strategy (excess)", linewidth=1.5)
axes[0].set_ylabel("Cumulative Return")
axes[0].set_title("Strategy Cumulative Excess Return vs Benchmark")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Drawdown
cumret = (1 + report["return"]).cumprod()
drawdown = cumret / cumret.cummax() - 1
axes[1].fill_between(drawdown.index, drawdown, 0, alpha=0.4, color="red")
axes[1].set_ylabel("Drawdown")
axes[1].set_title("Drawdown")
axes[1].grid(True, alpha=0.3)

# Turnover
if "turnover" in report.columns:
    axes[2].bar(report.index, report["turnover"], alpha=0.5, width=1)
    axes[2].set_ylabel("Turnover")
    axes[2].set_title("Daily Turnover")
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4b. IC (Information Coefficient) Over Time

The **IC** measures the correlation between predicted returns and actual returns
each day. Key metrics:
- **IC > 0.03** is generally considered a useful signal
- **ICIR** (IC / std of IC) > 0.5 indicates a consistent signal
- **Rank IC** is more robust to outliers

In [ ]:
ic = results["ic_analysis"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# IC time series
axes[0].bar(ic.index, ic["IC"], alpha=0.5, width=1, label="Daily IC")
axes[0].axhline(y=ic["IC"].mean(), color="red", linestyle="--", label=f"Mean IC = {ic['IC'].mean():.4f}")
axes[0].set_ylabel("IC")
axes[0].set_title("Information Coefficient (IC) Over Time")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative IC
axes[1].plot(ic.index, ic["IC"].cumsum(), linewidth=1.5)
axes[1].set_ylabel("Cumulative IC")
axes[1].set_title("Cumulative IC — a rising line means consistent predictive power")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nIC Mean:      {ic['IC'].mean():.4f}")
print(f"IC Std:       {ic['IC'].std():.4f}")
print(f"ICIR:         {ic['IC'].mean() / ic['IC'].std():.4f}")
print(f"Rank IC Mean: {ic['Rank IC'].mean():.4f}")

### 4c. Monthly Returns Heatmap

A heatmap of monthly returns helps you spot seasonal patterns and identify
when the strategy performs well or poorly.

In [ ]:
# Monthly returns heatmap
monthly = report["return"].resample("ME").sum()
monthly_pivot = pd.DataFrame({
    "year": monthly.index.year,
    "month": monthly.index.month,
    "return": monthly.values,
}).pivot(index="year", columns="month", values="return")

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly_pivot.columns = [month_names[m-1] for m in monthly_pivot.columns]

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(monthly_pivot.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(monthly_pivot.columns)))
ax.set_xticklabels(monthly_pivot.columns)
ax.set_yticks(range(len(monthly_pivot.index)))
ax.set_yticklabels(monthly_pivot.index)
ax.set_title("Monthly Excess Returns")
plt.colorbar(im, ax=ax, label="Return")

# Annotate cells
for i in range(len(monthly_pivot.index)):
    for j in range(len(monthly_pivot.columns)):
        val = monthly_pivot.values[i, j]
        if not pd.isna(val):
            ax.text(j, i, f"{val:.1%}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## Step 5 — Key Takeaways

After reviewing the results above, consider:

- **Is the IC positive and consistent?** An ICIR above 0.5 suggests a
  reliable signal.
- **How does the strategy compare to SPY?** A rising cumulative excess return
  means the model adds value over buy-and-hold.
- **How deep are the drawdowns?** Max drawdown tells you the worst-case pain.
- **Is turnover reasonable?** High turnover increases transaction costs.

### Next Steps
- Try different hyperparameters in `configs/lgbm_alpha158.yaml`
- Swap LightGBM for a neural network (e.g., `qlib.contrib.model.pytorch_alstm`)
- Add custom features beyond Alpha158
- Test on different stock universes or time periods